In [ ]:
import torch
torch.manual_seed(42)

def FFT_boundary_conv(x, kernel, padding):
    """verification the correctness of FFT-Boundary method"""
    H, W = x.shape[-2:]
    kernel_H, kernel_W = kernel.shape[-2:]
    y_size = (H + 2 * padding - kernel_H + 1, W + 2 * padding - kernel_W + 1)
    fft_x = torch.fft.fft2(x)
    kernel_flip = torch.flip(kernel, dims=(-2, -1))
    fft_kernel = torch.fft.fft2(kernel_flip, s=(H, W))
    fft_y = torch.einsum('dcuv, bcuv->bduv', fft_kernel, fft_x)

    if padding > 0:
        def get_boundary_mask(xx, yy):
            """the boundary active set B(xx,yy) (Fig.S1 gives the example of B(0,1) with padding=0)"""
            top_row = max(xx-padding, 0)
            left_col = max(yy-padding, 0)
            bottom_row = max(kernel_H-1-xx-padding, 0)
            right_col = max(kernel_W-1-yy-padding, 0)
            bound = torch.zeros((H, W), dtype=torch.bool)
            bound[:top_row,:] = True
            bound[:,:left_col] = True
            if bottom_row>0:  bound[-bottom_row:,:] = True
            if right_col>0:  bound[:,-right_col:] = True
            return bound

        DFT_H = torch.fft.fft(torch.eye(H, dtype=x.dtype, device=x.device), dim=0)
        DFT_W = torch.fft.fft(torch.eye(W, dtype=x.dtype, device=x.device), dim=0)
        G = torch.zeros_like(fft_y)
        for i in range(kernel_H):
            for j in range(kernel_W):
                mask = get_boundary_mask(kernel_H - i - 1, kernel_W - j - 1)
                phase = DFT_H[:, i, None] * DFT_W[:, j]  # u, v
                fft_boundary = torch.fft.fft2(x * mask)  # bs, c1, u, v
                weights = kernel_flip[:, :, i, j].to(fft_boundary.dtype)  # c2, c1
                G += torch.einsum('dc,bcuv->bduv', weights, fft_boundary) * phase
        fft_y -= G

    y = torch.fft.ifft2(fft_y).real
    y = torch.roll(y, shifts=(-padding, -padding), dims=(-2, -1))
    
    return y[..., -y_size[-2]:, -y_size[-1]:]

X_SHAPE = (1,1,7,7)  # (BS, C1, H1, W1)
KERNEL_SHAPE = (1, X_SHAPE[1], 5, 5) # (C2, C1, kernel_H, kernel_W)
PADDING: int = 1
assert 2*PADDING <= min(KERNEL_SHAPE[-2], KERNEL_SHAPE[-1])-1 # We only support y_size <= x_size

x = torch.rand(X_SHAPE, dtype=torch.float64)    # (BS, C1, H1, W1)
kernel = torch.rand(KERNEL_SHAPE, dtype=torch.float64)  # (C2, C1, kernel_H, kernel_W)
y = torch.nn.functional.conv2d(x, kernel, padding=PADDING, stride=1, dilation=1) # (BS, C2, H2, W2)

other_y = FFT_boundary_conv(x, kernel, PADDING)  # get the result using FFT methods
print(f"FFT_boundary conv result: max_absolute_error = {torch.max(torch.abs(y - other_y))}\n")

FFT_boundary conv result: max_absolute_error = 2.220446049250313e-15



In [14]:
import torch
torch.manual_seed(42)

def FFT_padded_conv(x, kernel, y_shape, PADDING):
    """verification the corectness of FFT-Padded method"""
    pad_x = torch.nn.ZeroPad2d((PADDING, PADDING, PADDING, PADDING))
    x_padded = pad_x(x.to(torch.float64))
    fft_x = torch.fft.fft2(x_padded, dim=[-2, -1]) 
    kernel_flip = torch.flip(kernel, dims=[-2, -1])
    fft_kernel_flip = torch.fft.fft2(kernel_flip, s=fft_x.shape[-2:], dim=[-2, -1])  # pad at bottom and right by default
    other_fft_y = torch.einsum('bcuv,dcuv->bduv', fft_x, fft_kernel_flip)
    other_y = torch.fft.ifft2(other_fft_y).real
    return other_y[:,:,-y_shape[-2]:, -y_shape[-1]:]

X_SHAPE = (1,1,5,5)  # (BS, C1, H1, W1)
KERNEL_SHAPE = (1,X_SHAPE[1],3,3) # (C2, C1, kernel_H, kernel_W)
PADDING=1
x = torch.rand(X_SHAPE, dtype=torch.float64)    # (BS, C1, H1, W1)
kernel = torch.rand(KERNEL_SHAPE, dtype=torch.float64)  # (C2, C1, kernel_H, kernel_W)
y = torch.nn.functional.conv2d(x, kernel, padding=PADDING, stride=1, dilation=1) # (BS, C2, H2, W2)

other_y = FFT_padded_conv(x, kernel, y.shape, PADDING)  # get the result using FFT methods
assert y.shape == other_y.shape, f"y.shape = {y.shape}, other_y.shape = {other_y.shape}"
print(f"FFT_padded conv result: max_absolute_error = {torch.max(torch.abs(y - other_y))}\n")

FFT_padded conv result: max_absolute_error = 4.440892098500626e-16



In [ ]:
x = torch.tensor([[[[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]]])   # (BS, C1, H1, W1)
kernel = torch.tensor([[[[1,0,0],[0,1,1],[0,0,0]]]])  # (C2, C1, kernel_H, kernel_W)
y = torch.nn.functional.conv2d(x, kernel, padding=0, stride=1, dilation=1) # (BS, C2, H2, W2)
print(x)
print(kernel)
print(y)
print('----------')
fft_x = torch.fft.fft2(x, dim=[-2, -1]) 
kernel_flip = torch.flip(kernel, dims=[-2, -1])
fft_kernel_flip = torch.fft.fft2(kernel_flip, s=fft_x.shape[-2:], dim=[-2, -1])  # pad at bottom and right by default
print(torch.fft.ifft2(fft_kernel_flip).real)
other_fft_y = torch.einsum('bcuv,dcuv->bduv', fft_x, fft_kernel_flip)
other_y = torch.fft.ifft2(other_fft_y).real
print(other_y)
def circular_convolution(f: torch.Tensor, g: torch.Tensor):
    assert f.shape == g.shape
    H, W = f.shape
    result = torch.zeros_like(f)

    # result[i, j] = sum f[m, n] * g[(i-m) % H, (j-n) % W]
    for i in range(H):
        for j in range(W):
            for m in range(H):
                for n in range(W):
                    result[i, j] += f[m, n] * g[(i - m) % H, (j - n) % W]
    return result
circular_convolution(torch.fft.ifft2(fft_kernel_flip).real[0,0,:,:], x[0,0,:,:])

tensor([[[[ 1,  2,  3,  4],
          [ 5,  6,  7,  8],
          [ 9, 10, 11, 12],
          [13, 14, 15, 16]]]])
tensor([[[[1, 0, 0],
          [0, 1, 1],
          [0, 0, 0]]]])
tensor([[[[14, 17],
          [26, 29]]]])
----------
tensor([[[[0., 0., 0., 0.],
          [1., 1., 0., 0.],
          [0., 0., 1., 0.],
          [0., 0., 0., 0.]]]])
tensor([[[[40., 39., 38., 41.],
          [20., 19., 18., 21.],
          [16., 15., 14., 17.],
          [28., 27., 26., 29.]]]])


In [ ]:
|